# Build a Mixture-of-Experts Transformer: quickstart

A dense transformer sends every token through the same feed-forward network. A **Mixture-of-Experts (MoE)** transformer
keeps many smaller feed-forward networks, the *experts*, and a *router* that sends each token to only a few of them. The
model stores more weights, while each token pays for only the experts it visits.

This notebook runs the course's reference solution: the tests for the first twelve stages, the router and the two
load-balancing methods on toy inputs, a short MoE vs dense training run, a look at which characters go to which
expert, and a router that collapses and recovers. Everything runs on a CPU (a Colab CPU runtime is enough) in about
ten minutes.

In [ ]:
# Setup: clone the course and install what Colab doesn't already have (torch is preinstalled on Colab).
import os, sys
if os.path.basename(os.getcwd()) != "build-a-moe-transformer":
    if os.path.basename(os.getcwd()) == "notebooks" and os.path.exists("../course-definition.yml"):
        os.chdir("..")
    else:
        if not os.path.isdir("build-a-moe-transformer"):
            !git clone -q --depth 1 https://github.com/albertlieyingadrian/build-a-moe-transformer
        os.chdir("build-a-moe-transformer")
!{sys.executable} -m pip install -q -r requirements.txt
!make data > /dev/null
os.environ["PYTHON"] = sys.executable
P = "solutions/python/17-mtp/code/your_program.sh"
print(os.getcwd())

## 1. The first twelve stages' tests

Each stage adds one piece and has its own tests. `solutions/python/NN-<slug>/code/` is the reference program right
after stage NN. The tests run a program through its command line, the way an automated grader would. Stages 01-12
build the pieces (tokenizer, a dense GPT, a SwiGLU expert, the router, dispatch, capacity, the two balancing methods,
the shared expert, fine-grained experts and latent attention), and their tests take a few seconds.

In [ ]:
!make test-upto STAGE=12 PY={sys.executable}

## 2. The router, and two ways to balance it

A router scores every expert for every token and keeps the top k. Here, 8 experts and top-2: each token gets two
experts and two weights that sum to 1.

In [ ]:
!./{P} router

Routers tend to pile tokens onto a few experts. **max load** is the busiest expert's share times the number of experts:
1.0 is even, 4.0 is the worst top-2 allows. The classic fix is an *auxiliary loss* that penalises lopsided routing
(stage 08). DeepSeek-V3's fix (stage 09) is a *bias* per expert that is nudged down when the expert is busy and up when
it is idle. The bias changes which experts are picked and never the weights, so it adds nothing to the loss.

In [ ]:
!./{P} aux-loss
print()
!./{P} bias-balance

## 3. MoE vs dense at the same active parameters

`dense` has one feed-forward network as wide as the weights one MoE token uses (2 routed experts + 1 shared). `moe-bias`
has 8 routed experts and 1 shared, top-2, balanced by the bias. Same work per token; the MoE stores about 2.4 times the
weights. This is a short run (600 steps, a smaller model) so it fits a notebook; the README has the full 3,000-step
runs over three seeds.

In [ ]:
import time
t = time.time()
SMALL = "--steps 600 --layers 2 --n-embd 64 --expert-hidden 64 --block-size 64"
!./{P} compare data/tinyshakespeare.txt {SMALL} --variants dense,moe-bias --out-dir runs/colab
print(f"took {time.time() - t:.0f} s")

In [ ]:
from IPython.display import Image, display
display(Image("runs/colab/compare_s0.png"))

## 4. Who goes where

Route the validation text through the trained MoE and count, for each expert, which kinds of characters it gets. The
number to read is **token → expert**: how much of the uncertainty about a token's expert is removed by knowing the
token (0 = routing ignores the token, 1 = the token decides). The shuffled control keeps both distributions and breaks
the link, so it should be near 0.

In [ ]:
!./{P} route-lab data/tinyshakespeare.txt --ckpt runs/colab/moe-bias_s0.pt --out-dir runs/colab
display(Image("runs/colab/routing_heatmap.png"))

## 5. Routing collapse, and recovery

Give expert 0 a fixed head start in every router, train, and watch the worst layer's max load. With no balancing
(`none-skew`) expert 0 stays in every token's top-2. The aux loss and the bias both pull it back; `none` is the same
model without the head start.

In [ ]:
!./{P} collapse data/tinyshakespeare.txt --steps 300 --layers 2 --n-embd 64 --expert-hidden 64 --block-size 64 --out-dir runs/colab
display(Image("runs/colab/collapse_s0.png"))

## Where to go next

- Build it yourself: copy `starter_templates/python/code`, open `app/main.py`, and follow the stage descriptions in
  [`stage_descriptions/`](https://github.com/albertlieyingadrian/build-a-moe-transformer/tree/main/stage_descriptions), starting with
  [stage 01](https://github.com/albertlieyingadrian/build-a-moe-transformer/blob/main/stage_descriptions/base-01-tokenize.md).
- Switching experts off one at a time (stage 15) and multi-token prediction (stage 17) are one command each:
  `ablate` and `mtp` in `solutions/python/17-mtp/code/your_program.sh`.
- The same model in one file: [nanomoe-transformer](https://github.com/albertlieyingadrian/nanomoe-transformer).